In [1]:
import sys
import os

# Add the repo root to path so 'src' is findable
sys.path.insert(0, os.path.abspath('./Barak-Cajiao-Trabecular-Bone-Analysis'))


In [2]:
print(os.getcwd())

/home/jovyan


In [3]:
import src.config.generation_settings
import src.config.paths as paths
import src.pipeline.run_pipeline as run_pipeline
import ipywidgets as widgets
from IPython.display import display, clear_output

printer_box = widgets.Dropdown(options=["BMF", "Formlabs"], value="BMF", description="Printer:")
method_box = widgets.Dropdown(options=["Tension", "Compression"], value="Tension", description="Method:")

medial_lateral = widgets.Checkbox(value=True, description="Medial-Lateral")
cranial_caudal = widgets.Checkbox(value=True, description="Cranial-Caudal")
proximal_distal = widgets.Checkbox(value=True, description="Proximal-Distal")
median_graph = widgets.Checkbox(value=True, description="Median Graph")

orientation_row = widgets.HBox([medial_lateral, cranial_caudal, proximal_distal])

generate_btn = widgets.Button(description="Generate Graph", button_style="primary")
plot_output = widgets.Output()  # ← this is your canvas replacement

def on_generate(b):
    generate_btn.disabled = True
    generate_btn.description = "Running..."
    with plot_output:
        clear_output(wait=True)
        # call your run_pipeline here, it renders matplotlib into this output
        run_pipeline.generate_graph(current_analysis)
    generate_btn.disabled = False
    generate_btn.description = "Generate Graph"

generate_btn.on_click(on_generate)

tab1 = widgets.VBox([printer_box, method_box, orientation_row, median_graph, generate_btn, plot_output])

In [4]:
def build_checkbox_tree(root_path, label):
    items = []
    checked = {}

    def walk(path, indent=0):
        if os.path.isdir(path):
            name = os.path.basename(path)
            # Directory shown as a plain label, not a checkbox
            lbl = widgets.Label(value=("  " * indent) + f"📁 {name}",
                                layout=widgets.Layout(width='300px'))
            items.append(lbl)
            for child in sorted(os.listdir(path)):
                walk(os.path.join(path, child), indent + 1)
        else:
            name = os.path.basename(path)
            cb = widgets.Checkbox(value=True, description=("  " * indent) + name,
                                  layout=widgets.Layout(width='300px'))
            checked[path] = cb
            items.append(cb)

    walk(root_path)

    master = widgets.Checkbox(value=True, description=f"✓ {label} (all)",
                              layout=widgets.Layout(width='300px'))

    def on_master_change(change):
        for cb in checked.values():
            cb.value = change['new']

    master.observe(on_master_change, names='value')

    return widgets.VBox([master, *items]), checked

cc_tree, checked_CC = build_checkbox_tree(paths.PROCESSED_DATA / 'bmf' / 'tension' / 'cranial-caudal', "Cranial-Caudal")
ml_tree, checked_ML = build_checkbox_tree(paths.PROCESSED_DATA / 'bmf' / 'tension' / 'medial-lateral', "Medial-Lateral")
pd_tree, checked_PD = build_checkbox_tree(paths.PROCESSED_DATA / 'bmf' / 'tension' / 'proximal-distal', "Proximal-Distal")

tab2 = widgets.HBox([cc_tree, ml_tree, pd_tree])

In [5]:
path = paths.PROCESSED_DATA / 'bmf' / 'tension' / 'cranial-caudal'
print(os.path.isdir(path))
print(os.listdir(path))

True
['BMF_Beam_Two9-2024-04-25-08-40-07.xlsx', 'BMF_Beam_Two8-2024-04-25-08-25-53.xlsx', 'BMF_Beam_Two3-2024-04-01-10-55-45.xlsx', 'BMF_Beam_Two12-2024-04-25-09-10-27.xlsx', 'BMF_Beam_Two10-2024-04-25-08-50-11.xlsx', 'BMF_Beam_Two1-2024-03-28-10-10-50.xlsx', 'BMF_Beam_Two11-2024-04-25-09-02-03.xlsx', 'BMF_Beam_Two5-2024-04-01-11-41-01.xlsx', 'BMF_Beam_Two2-2024-03-28-11-19-10.xlsx', 'BMF_Beam_Two6-2024-04-01-11-49-35.xlsx']


In [6]:
tabs = widgets.Tab(children=[tab1, tab2])
tabs.set_title(0, "Analysis")
tabs.set_title(1, "Files")

display(tabs)